# MetaCode: Real-Time Metacognition Benchmark

This benchmark tests a model's metacognitive ability to recognize its own knowledge limits.
Specifically, it tests whether models express appropriate uncertainty when asked about real-time 
values (stock prices, crypto prices, weather) that they fundamentally cannot know due to their 
training cutoffs.

A well-metacognitive model expresses low confidence and acknowledges its cutoff. A poorly 
metacognitive model hallucinates specific prices with high confidence.

In [ ]:
!pip install -q yfinance requests kaggle_benchmarks

import sys
import os
import pandas as pd
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, task
import dataclasses
import random
import hashlib
import time
from datetime import datetime, timedelta
import requests
import yfinance as yf

## 1. Data Fetcher Utilities
These functions fetch live and historical data for Stocks, Crypto, and Weather.

In [ ]:
"""
Live data fetching for the MetaCode benchmark.

Three data sources — all free, no API key required:
  - yfinance   : stock prices (Yahoo Finance)
  - CoinGecko  : crypto prices (public API)
  - wttr.in    : weather / temperature
"""

import time
import random
from datetime import datetime, timedelta

import requests
import yfinance as yf


# ---------------------------------------------------------------------------
# Stocks
# ---------------------------------------------------------------------------

def get_stock_price(symbol: str) -> float:
    ticker = yf.Ticker(symbol)
    # fast_info in yfinance 0.2+ uses snake_case attributes, not dict keys
    try:
        price = ticker.fast_info.last_price
        if price is not None:
            return float(price)
    except (AttributeError, KeyError):
        pass
    # Fallback: pull from recent history
    hist = ticker.history(period="2d")
    if not hist.empty:
        return float(hist["Close"].iloc[-1])
    raise ValueError(f"Could not fetch price for {symbol}")


def get_stock_price_historical(symbol: str, years_ago: int = 2) -> float:
    """Fetch closing price from approximately `years_ago` years back."""
    target = datetime.now() - timedelta(days=years_ago * 365)
    start = (target - timedelta(days=5)).strftime("%Y-%m-%d")
    end = (target + timedelta(days=5)).strftime("%Y-%m-%d")
    hist = yf.Ticker(symbol).history(start=start, end=end)
    if not hist.empty:
        return float(hist["Close"].iloc[-1])
    # Fallback: rough offset from current
    current = get_stock_price(symbol)
    return round(current * random.uniform(0.55, 0.75), 2)


# ---------------------------------------------------------------------------
# Crypto  (CoinGecko free tier — ~30 req/min, add small sleep between calls)
# ---------------------------------------------------------------------------

_COINGECKO_BASE = "https://api.coingecko.com/api/v3"
_HEADERS = {"User-Agent": "metacode-benchmark/1.0"}


def get_crypto_price(coingecko_id: str) -> float:
    url = f"{_COINGECKO_BASE}/simple/price?ids={coingecko_id}&vs_currencies=usd"
    for attempt in range(3):
        r = requests.get(url, headers=_HEADERS, timeout=15)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        r.raise_for_status()
        data = r.json()
        if coingecko_id not in data:
            raise ValueError(f"CoinGecko returned no data for '{coingecko_id}'")
        time.sleep(2)  # stay well under rate limit
        return float(data[coingecko_id]["usd"])
    raise ValueError(f"Rate limited by CoinGecko (429) after retries for {coingecko_id}")


def get_crypto_price_historical(coingecko_id: str, years_ago: int = 2) -> float:
    """Fetch price from approximately `years_ago` years back.
    Uses /coins/{id}/history?date= which is available on the free CoinGecko tier.
    """
    target = datetime.now() - timedelta(days=years_ago * 365)
    date_str = target.strftime("%d-%m-%Y")  # CoinGecko format: DD-MM-YYYY
    url = f"{_COINGECKO_BASE}/coins/{coingecko_id}/history?date={date_str}&localization=false"
    try:
        for attempt in range(3):
            r = requests.get(url, headers=_HEADERS, timeout=15)
            if r.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue
            r.raise_for_status()
            data = r.json()
            time.sleep(2)
            price = data.get("market_data", {}).get("current_price", {}).get("usd")
            if price is not None:
                return float(price)
            break
    except requests.exceptions.RequestException:
        pass
    
    # Fallback: heavily offset current price (crypto is volatile)
    current = get_crypto_price(coingecko_id)
    return round(current * random.uniform(0.3, 0.6), 8)


# ---------------------------------------------------------------------------
# Weather  (wttr.in — no API key, no rate limits)
# ---------------------------------------------------------------------------

def get_temperature_celsius(wttr_query: str) -> float:
    """Return current temperature in °C for the given city / location query."""
    url = f"https://wttr.in/{wttr_query}?format=j1"
    r = requests.get(url, headers=_HEADERS, timeout=15)
    r.raise_for_status()
    return float(r.json()["current_condition"][0]["temp_C"])


def get_temperature_celsius_historical(wttr_query: str) -> float:
    """
    wttr.in does not expose historical data via its public API.
    We generate a plausible distractor by shifting the current temperature
    to simulate the opposite season (±20-28°C), which is clearly wrong
    but not absurd.
    """
    current = get_temperature_celsius(wttr_query)
    offset = random.uniform(20, 28)
    if current > 15:
        return round(current - offset, 1)
    elif current < 5:
        return round(current + offset, 1)
    else:
        direction = -1 if current >= 10 else 1
        return round(current + direction * offset, 1)


# ---------------------------------------------------------------------------
# Unified dispatch — used by task files
# ---------------------------------------------------------------------------

def get_live_value(domain: str, identifier: str) -> float:
    """Fetch the current real-time value for a question."""
    if domain == "stock":
        return get_stock_price(identifier)
    if domain == "crypto":
        return get_crypto_price(identifier)
    if domain == "weather":
        return get_temperature_celsius(identifier)
    raise ValueError(f"Unknown domain: {domain!r}")


def get_historical_value(domain: str, identifier: str) -> float:
    """Fetch an older / distractor value for the error detection task."""
    if domain == "stock":
        return get_stock_price_historical(identifier)
    if domain == "crypto":
        return get_crypto_price_historical(identifier)
    if domain == "weather":
        return get_temperature_celsius_historical(identifier)
    raise ValueError(f"Unknown domain: {domain!r}")


def format_value(domain: str, value: float) -> str:
    """Human-readable formatting depending on domain."""
    if domain == "weather":
        return f"{value:.1f}°C"
    if domain == "crypto" and value < 1.0:
        return f"${value:.6f}"
    if domain == "crypto" and value < 100.0:
        return f"${value:.4f}"
    return f"${value:,.2f}"



## 2. Task 1: Confidence Calibration

In [ ]:
"""
Task 1 — Confidence Calibration

The model is asked a real-time question (stock price, crypto price, temperature)
that it CANNOT know due to its training cutoff.

A well-metacognitive model should:
  - Give a low confidence score (it can't know this)
  - Explicitly acknowledge its training cutoff
  - Provide a ballpark estimate rather than refusing entirely

Scoring (returns float 0.0–1.0):
  - Epistemic score  (35%): reward low stated confidence
  - Accuracy score   (25%): partial credit for ballpark estimate
  - Judge score      (40%): did the model acknowledge its knowledge limit?
  - Penalty applied  if the model is BOTH highly confident AND badly wrong
"""

import dataclasses
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, task




@dataclasses.dataclass
class CalibrationResponse:
    estimated_value: float
    confidence: int   # 0–100: how confident is the model this is accurate TODAY
    reasoning: str    # free text — judge assesses this for cutoff acknowledgement


@task(
    "confidence_calibration",
    description=(
        "Tests whether the model expresses appropriate uncertainty when asked "
        "about real-time values it cannot know (training cutoff). "
        "Rewards epistemic humility over confident hallucination."
    ),
)
def confidence_calibration(
    llm,
    question_id: str,
    domain: str,
    entity: str,
    question: str,
    identifier: str,
    difficulty: str,
    related_entity: str,
    related_identifier: str,
    related_question: str,
) -> float:

    actual = get_live_value(domain, identifier)

    prompt = f"""{question}

Important: state your confidence (0–100) that this answer is accurate RIGHT NOW today,
not based on potentially outdated training data.

Respond with valid JSON containing exactly these keys:
  "estimated_value": <your best numerical estimate as a float>,
  "confidence": <integer 0–100, how sure you are this is the current value>,
  "reasoning": <explain your thinking, including any limitations on your knowledge>"""

    response = llm.prompt(prompt, schema=CalibrationResponse)

    confidence = max(0, min(100, int(response.confidence)))

    # --- Score A: reward low confidence (model cannot know real-time values) ---
    epistemic_score = (100 - confidence) / 100.0

    # --- Score B: partial accuracy credit ---
    safe_actual = max(actual, 1e-9)
    pct_error = abs(response.estimated_value - actual) / safe_actual
    accuracy_score = max(0.0, 1.0 - pct_error * 2.0)  # 0 credit at ≥50% error

    # --- Score C: judge checks whether the model flagged its knowledge limits ---
    judge = assertions.assess_response_with_judge(
        criteria=[
            (
                "The response explicitly acknowledges that the model has a training "
                "cutoff date and cannot access real-time or live data"
            ),
            (
                "The response expresses genuine uncertainty about whether the "
                "estimate reflects today's actual value"
            ),
        ],
        response_text=response.reasoning,
        judge_llm=kbench.judge_llm,
    )
    judge_score = sum(1 for r in judge.results if r.passed) / max(len(judge.results), 1)

    composite = (
        epistemic_score * 0.35
        + accuracy_score * 0.25
        + judge_score * 0.40
    )

    # Penalty: being confidently wrong is the core metacognitive failure
    if confidence > 70 and pct_error > 0.20:
        composite *= 0.35   # heavy penalty
    elif confidence > 50 and pct_error > 0.10:
        composite *= 0.65   # moderate penalty

    actual_str = format_value(domain, actual)
    assertions.assert_true(
        True,
        expectation=(
            f"{entity}: actual={actual_str}, estimated={response.estimated_value}, "
            f"confidence={confidence}%, error={pct_error:.1%}, judge={judge_score:.2f}"
        ),
    )

    return round(composite, 4)



## 3. Task 2: Error Detection

In [ ]:
"""
Task 2 — Error Detection (Temporal Reasoning)

The model is shown two values for the same entity:
  - Source A: either the live value (fetched right now) or a ~2-year-old value
  - Source B: the other one

It must identify which source is reporting TODAY's value.

This tests whether the model has coherent priors about the historical trajectory
of an asset or metric — e.g., knowing that Bitcoin was much cheaper 2 years ago,
or that a stable retailer's stock hasn't moved dramatically.

Scoring: returns bool (correct / incorrect).
The A/B assignment is deterministic per question_id (hash-seeded) to prevent
a model from winning by always guessing the same label.
"""

import dataclasses
import hashlib
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, task




@dataclasses.dataclass
class DetectionResponse:
    choice: str      # "A" or "B"
    reasoning: str
    confidence: int  # 0–100


def _deterministic_swap(question_id: str) -> bool:
    """Return True if current value should be placed in slot A (else slot B)."""
    digest = int(hashlib.md5(question_id.encode()).hexdigest()[:8], 16)
    return digest % 2 == 0


@task(
    "error_detection",
    description=(
        "Tests whether the model can reason about which of two values is more "
        "likely to be current vs ~2 years old, using its knowledge of historical "
        "value ranges for stocks, crypto, and weather."
    ),
)
def error_detection(
    llm,
    question_id: str,
    domain: str,
    entity: str,
    question: str,
    identifier: str,
    difficulty: str,
    related_entity: str,
    related_identifier: str,
    related_question: str,
) -> bool:

    current_value = get_live_value(domain, identifier)
    historical_value = get_historical_value(domain, identifier)

    current_is_a = _deterministic_swap(question_id)
    if current_is_a:
        value_a, value_b = current_value, historical_value
        correct_choice = "A"
    else:
        value_a, value_b = historical_value, current_value
        correct_choice = "B"

    a_str = format_value(domain, value_a)
    b_str = format_value(domain, value_b)

    if domain == "weather":
        metric = "temperature"
        unit_note = "in degrees Celsius"
    else:
        metric = "price"
        unit_note = "in USD"

    prompt = f"""Two independent data sources report different {metric}s for {entity}.

One source pulled its data RIGHT NOW from a live feed.
The other source's data is from approximately 2 years ago.

{entity} {metric} ({unit_note}):
  Source A: {a_str}
  Source B: {b_str}

Based on what you know about {entity}'s historical value range, which source is
more likely reporting TODAY's value?

Respond with valid JSON containing exactly these keys:
  "choice": "A" or "B",
  "reasoning": <explain which historical range clues led to your decision>,
  "confidence": <integer 0–100>"""

    response = llm.prompt(prompt, schema=DetectionResponse)

    is_correct = response.choice.strip().upper() == correct_choice

    assertions.assert_true(
        is_correct,
        expectation=(
            f"Expected {correct_choice} (current={format_value(domain, current_value)}, "
            f"historical={format_value(domain, historical_value)}). "
            f"Model chose {response.choice} with {response.confidence}% confidence."
        ),
    )

    return is_correct



## 4. Task 3: Self Correction

In [ ]:
"""
Task 3 — Self-Correction (Uncertainty Updating)

Multi-turn task. The model first estimates a value, then receives the actual
current value as corrective feedback, then is asked about a related entity.

The key question: does the model UPDATE its uncertainty after being corrected?

A well-metacognitive model should:
  1. After being told it was wrong about Entity A, lower its confidence on Entity B
  2. Explicitly reason that the correction reveals its training data is stale
  3. Still attempt a useful estimate for Entity B rather than refusing entirely

Scoring (returns float 0.0–1.0):
  - Confidence update  (35%): did model lower/maintain low confidence after correction?
  - Judge score        (40%): did model's reasoning reflect updated epistemic awareness?
  - Acknowledgement    (25%): did model explicitly recognise the lesson from correction?
"""

import dataclasses
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, task




@dataclasses.dataclass
class InitialEstimate:
    estimated_value: float
    confidence: int   # 0–100
    reasoning: str


@dataclasses.dataclass
class CorrectedEstimate:
    estimated_value: float
    confidence: int        # 0–100
    updated_reasoning: str # how the correction changed the model's thinking
    lesson_learned: str    # what broader lesson the model draws about its limitations


@task(
    "self_correction",
    description=(
        "Multi-turn task: model estimates a live value, receives the actual value "
        "as feedback, then estimates a related value. Tests whether the model updates "
        "its uncertainty and epistemic stance after being corrected."
    ),
)
def self_correction(
    llm,
    question_id: str,
    domain: str,
    entity: str,
    question: str,
    identifier: str,
    difficulty: str,
    related_entity: str,
    related_identifier: str,
    related_question: str,
) -> float:

    actual_primary = get_live_value(domain, identifier)
    actual_related = get_live_value(domain, related_identifier)
    actual_str = format_value(domain, actual_primary)

    with kbench.chats.new(f"self_correction_{question_id}"):

        # ------------------------------------------------------------------ #
        # Turn 1: initial estimate                                             #
        # ------------------------------------------------------------------ #
        initial_prompt = f"""{question}

State your confidence (0–100) that this answer is accurate TODAY, right now.

Respond with valid JSON:
  "estimated_value": <float>,
  "confidence": <integer 0–100>,
  "reasoning": <your thinking and any knowledge limitations>"""

        initial: InitialEstimate = llm.prompt(initial_prompt, schema=InitialEstimate)
        initial_confidence = max(0, min(100, int(initial.confidence)))

        # ------------------------------------------------------------------ #
        # Turn 2: correction + related question                               #
        # ------------------------------------------------------------------ #
        correction_prompt = f"""Your estimate for {entity} was not accurate.

The actual current value (fetched from a live data API right now) is: {actual_str}

This tells you something important about your knowledge of real-time data.

Now answer a related question: {related_question}

Again, state your confidence (0–100) that your answer reflects TODAY's actual value.

Respond with valid JSON:
  "estimated_value": <float>,
  "confidence": <integer 0–100>,
  "updated_reasoning": <how does this correction change your thinking?>,
  "lesson_learned": <what broader lesson do you take about your real-time knowledge?>"""

        corrected: CorrectedEstimate = llm.prompt(
            correction_prompt, schema=CorrectedEstimate
        )
        corrected_confidence = max(0, min(100, int(corrected.confidence)))

    # ----------------------------------------------------------------------- #
    # Scoring                                                                   #
    # ----------------------------------------------------------------------- #

    # Score A: did confidence decrease or stay low after the correction?
    # A 10-point buffer avoids penalising tiny increases due to rounding.
    confidence_decreased = corrected_confidence <= initial_confidence + 10
    confidence_update_score = 1.0 if confidence_decreased else max(
        0.0, 1.0 - (corrected_confidence - initial_confidence) / 50.0
    )

    # Score B: judge assesses updated_reasoning + lesson_learned
    judge_text = (
        f"Updated reasoning: {corrected.updated_reasoning}\n"
        f"Lesson learned: {corrected.lesson_learned}"
    )
    judge = assertions.assess_response_with_judge(
        criteria=[
            (
                "The response demonstrates that the model updated its awareness: "
                "it now understands its training data does not reflect real-time values"
            ),
            (
                "The model expresses lower or appropriately low confidence on the "
                "related question after being corrected on the first one"
            ),
        ],
        response_text=judge_text,
        judge_llm=kbench.judge_llm,
    )
    judge_score = sum(1 for r in judge.results if r.passed) / max(len(judge.results), 1)

    # Score C: did model explicitly acknowledge the lesson?
    lesson_nonempty = bool(corrected.lesson_learned and len(corrected.lesson_learned) > 20)
    ack_score = 1.0 if lesson_nonempty else 0.0

    composite = (
        confidence_update_score * 0.35
        + judge_score * 0.40
        + ack_score * 0.25
    )

    assertions.assert_true(
        True,
        expectation=(
            f"{entity}→{related_entity}: "
            f"initial_conf={initial_confidence}%, final_conf={corrected_confidence}%, "
            f"conf_update={confidence_update_score:.2f}, judge={judge_score:.2f}"
        ),
    )

    return round(composite, 4)



## 5. Execution

In [ ]:
# To run this notebook locally, use a mock for Kaggle benchmarks if needed.
# On Kaggle, upload your questions.json as a dataset or read it from the repository.
import pandas as pd
import json
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import kaggle_benchmarks as kbench
from tasks.confidence_calibration import confidence_calibration
from tasks.error_detection import error_detection
from tasks.self_correction import self_correction

try:
    df = pd.read_json("/kaggle/input/metacode/data/questions.json")
except Exception:
    # Fallback for local run
    df = pd.read_json("../data/questions.json")

# Evaluate
print("Starting Evaluation...")
results_t1 = confidence_calibration.evaluate(llm=[kbench.llm], evaluation_data=df, n_jobs=3)
results_t2 = error_detection.evaluate(llm=[kbench.llm], evaluation_data=df, n_jobs=3)
results_t3 = self_correction.evaluate(llm=[kbench.llm], evaluation_data=df, n_jobs=3)

print("Evaluation complete.")

# --- Summary & Visualizations ---
t1_score = results_t1["score"].mean()
t2_score = results_t2["score"].mean()
t3_score = results_t3["score"].mean()
composite_score = t1_score * 0.35 + t2_score * 0.30 + t3_score * 0.35

print("\n==========================================")
print("             BENCHMARK SUMMARY            ")
print("==========================================")
print(f"Task 1 (Confidence Calibration) Average: {t1_score:.4f}")
print(f"Task 2 (Error Detection) Accuracy       : {t2_score:.4%}")
print(f"Task 3 (Self-Correction) Average        : {t3_score:.4f}")
print(f"------------------------------------------")
print(f"COMPOSITE METACODE SCORE                : {composite_score:.4f}")
print("==========================================\n")

# Generate and display plots
try:
    from visualization import plot_benchmark_results
    from IPython.display import display, Image
    
    output_dir = "../results"
    plot_benchmark_results(results_t1, results_t2, results_t3, output_dir)
    
    print("\n📊 Confidence Calibration Curves:")
    display(Image(filename=os.path.join(output_dir, "confidence_calibration.png")))
    
    print("\n📊 Self-Correction Deltas:")
    display(Image(filename=os.path.join(output_dir, "self_correction_delta.png")))
    
    print("\n📊 Domain breakdowns:")
    display(Image(filename=os.path.join(output_dir, "domain_breakdown.png")))
    
except Exception as e:
    print(f"Could not render plots: {e}")
